# NVDA Robinhood Chain overnight measurement design

## tl;dr

This notebook compresses the validated 870,479 NVDA/USDG swaps into **20 economic observations**: one token session followed by one NVDA regular-market open. All 80 primary five-minute VWAP boundaries are present, although six sessions have a transparent thin-boundary warning. Full-return VWAP and last-trade sensitivity are highly correlated (0.994) but differ by a median 10.83 bp. Premarket contributes the largest median absolute segment move; deep overnight remains economically active.

The correlations below are exploratory and unstable at N=20. Predictive information content, price discovery, an attention mechanism, causality, and alpha are **not established**.

## Context & Methods

### Key assumptions

- America/New_York boundaries: POST 16:00–20:00, DEEP 20:00–04:00, PREMARKET 04:00–09:30. Weekend DEEP runs Friday 20:00 through Monday 04:00 so the segments partition the full session.
- Primary boundary prices are quote-notional-weighted executed-trade VWAPs in [16:00,16:05), [19:55,20:00), [03:55,04:00), and [09:25,09:30). Empty windows remain missing; no forward-fill or enlargement.
- Sensitivity prices use the last execution strictly before each boundary. Small five-minute pre-16:00 supplements were required because Phase 0 starts exactly at 16:00; all twenty were nonempty.
- Underlying target is log(next regular open / previous regular close), using raw Nasdaq Open and Close/Last—not adjusted close or a token reference quote.
- A five-minute boundary is warning-only 'thin' when it has fewer than 5 reconstructable swaps or under 500 USDG notional. No row is excluded.

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

project_root = Path.cwd() if (Path.cwd() / 'reports').exists() else Path.cwd().parent
result_dir = project_root / 'reports/robinhood_chain_pilot/nvda_20_session'
table = pd.read_csv(result_dir / 'nvda_overnight_session_measurements.csv')
diagnostics = json.loads((result_dir / 'diagnostics.json').read_text(encoding='utf-8'))
table.shape

## Data

### Validate the 20-row observation grain and identities

In [ ]:
assert len(table) == table['session_id'].nunique() == 20
assert table['return_decomposition_valid'].all()
assert table['return_decomposition_valid_last_trade'].all()
assert not table['missing_boundary_flag'].any()
assert table['nvda_previous_close'].notna().all()
assert table['nvda_next_open'].notna().all()
assert (table[['swap_count_post', 'swap_count_deep', 'swap_count_premarket']].sum(axis=1) == table['swap_count_full']).all()
table[['market_open_date', 'token_ret_full', 'token_ret_deep', 'nvda_gap_return', 'session_warning']].head()

### Inspect boundary depth and all warnings

In [ ]:
boundary_depth = pd.DataFrame(diagnostics['primary_boundary_depth']).T
warning_rows = table.loc[table['session_warning'].notna(), [
    'market_open_date', 'thin_boundary_detail', 'session_warning'
]]
boundary_depth, warning_rows

## Results

### Summary statistics at the economic observation grain

In [ ]:
summary_columns = [
    'token_ret_full', 'token_ret_post', 'token_ret_deep', 'token_ret_premarket',
    'nvda_gap_return', 'swap_count_full', 'notional_full', 'active_min_share_full',
    'price_reconstruction_rate', 'largest_execution_gap_minutes',
]
table[summary_columns].describe().T

### Compare primary VWAP and last-trade sensitivity

In [ ]:
pd.DataFrame(diagnostics['primary_vs_last_trade']).T

### Inspect extreme returns and high-activity sessions

In [ ]:
extremes = table.loc[table['token_ret_full'].abs().nlargest(6).index, [
    'market_open_date', 'token_ret_full', 'nvda_gap_return',
    'swap_count_full', 'notional_full', 'active_min_share_full',
]].sort_values('token_ret_full')
high_activity = table.nlargest(5, 'notional_full')[[
    'market_open_date', 'notional_full', 'swap_count_full',
    'token_ret_full', 'nvda_gap_return',
]]
extremes, high_activity

### Exploratory return–gap correlations

These are descriptive diagnostics only. N=20 is too small for stable inference, and FULL/POST/PREMARKET overlap with conventional extended-hours equity trading.

In [ ]:
correlations = pd.DataFrame(diagnostics['return_vs_gap_correlations_exploratory_only']).T
correlations[['n', 'pearson', 'spearman', 'leave_one_out_pearson_min', 'leave_one_out_pearson_max']]

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(10, 8), constrained_layout=True)
for axis, segment in zip(axes.flat, ('full', 'deep', 'post', 'premarket')):
    axis.scatter(table[f'token_ret_{segment}'] * 100, table['nvda_gap_return'] * 100, alpha=0.8)
    axis.axhline(0, color='0.6', linewidth=0.7)
    axis.axvline(0, color='0.6', linewidth=0.7)
    axis.set(title=segment.replace('_', ' ').title(), xlabel='Token log return (%)', ylabel='NVDA gap (%)')
fig.suptitle('NVDA token returns vs next regular-market gap — exploratory N=20')
plt.show()

## Takeaways

- **A — Boundary measurement:** all 80 primary VWAPs are measurable; six sessions have thin-boundary warnings and should remain visible in later robustness work.
- **B — Sensitivity:** full VWAP versus last-trade return differs by 10.83 bp median and 40.02 bp maximum; estimator choice is not optimized here.
- **C — Segment movement:** premarket has the largest median absolute log return (41.16 bp), followed by deep overnight (23.27 bp) and post-market (18.07 bp).
- **D — Deep activity:** median deep activity is 6,824 swaps, 5.31M USDG, and 93.9% active-minute coverage. Activity–absolute-move Spearman associations are all small (maximum magnitude 0.271); August 27 is active and extreme, but August 31 is more active by count with modest returns.
- **E — Target alignment:** all 20 targets use the prior regular close and next regular open from raw Nasdaq fields.
- **F — Correlations:** FULL Pearson/Spearman are 0.992/0.898; DEEP are 0.739/0.510. They are exploratory, partly contemporaneous, and not predictive evidence.
- **G — Dominance:** August 27 strongly dominates POST (Pearson falls from 0.807 to 0.260 when omitted). FULL stays between 0.980 and 0.995 under leave-one-out checks.
- **H — Expansion readiness:** the measurement contract is suitable for a later multi-token panel if frozen before expansion; predictive content and price discovery remain unestablished.